# DEE - Sachsen-Anhalt

In [1]:
import os
import re
from glob import glob
from pathlib import Path
from datetime import datetime

import zipfile
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

from camelsp import get_metadata

from camels_de1h import Station1h, get_input_path, get_nuts_id_from_provider_id

## Extract data

In [2]:
input_path = get_input_path("DEE")

files = glob(str(input_path / "Pegel Wasserstand-Durchfluss (gepruefte Daten)" / "Download" / "*.zip"))

if not Path(input_path / "Q_and_W").exists():
    os.makedirs(input_path / "Q_and_W", exist_ok=True)

    for f in files:
        filename = Path(f).stem
        os.makedirs(input_path / "Q_and_W" / filename, exist_ok=True)

        with zipfile.ZipFile(f, "r") as zip_ref:
            zip_ref.extractall(input_path / "Q_and_W" / filename)
    
    print("Discharge and water level data extracted.")
else:
    print("Discharge and water level data already extracted.")

Discharge and water level data already extracted.


## Parse data

The data is split into 2 parts:
* data before 2024, downloaded from the browser portal (https://gld.lhw-sachsen-anhalt.de/#) -> in UTC+0
* data after 2024, delivered via E-Mail -> in UTC+1

Next to the data files, there is also a metadata .csv file for each station.

Timezone (from `Impressum.pdf`):
> **Wichtiger Hinweis**: 
> Bitte beachten Sie, dass alle Datumsangaben mit Zeitangaben in UTC (Koordinierte Weltzeit) exportiert 
> werden. Dies betrifft u.a. die Daten zu Wasserstand und Durchfluss (Pegel-Daten). Hintergrund ist, dass lokale 
> Zeitangaben aufgrund der Umstellung von Sommer- auf Winterzeit nicht eindeutig zuzuordnen sind.

-> I guess this is UTC+0 then?

In [3]:
def parse_data_after_2024(file_path, station_id):
    """
    Extract data for a specific station from the 2024-2025 ZRX file.  
    Data is in UTC+1 timezone, we have to convert it to UTC by subtracting 1 hour.
    
    Args:
        file_path (str): Path to the .zrx file
        station_id (str): Station ID to extract (e.g., '579070')
    
    Returns:
        pd.DataFrame: DataFrame with datetime index and value/status columns
    """
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
    
    # Find the section for this station
    pattern = f"#SANR{station_id}\\|.*?(?=#ZRXPVERSION|$)"
    match = re.search(pattern, content, re.DOTALL)
    
    if not match:
        return pd.DataFrame()
    
    section = match.group(0)
    lines = section.split('\n')
    
    # Extract data
    data = []
    in_data = False
    
    for line in lines:
        line = line.strip()
        if "#LAYOUT" in line:
            in_data = True
            continue
        
        if in_data and line and not line.startswith('#') and re.match(r'^\d{14}', line):
            parts = line.split()
            if len(parts) >= 2:
                timestamp = datetime.strptime(parts[0], "%Y%m%d%H%M%S")
                value = float(parts[1]) if parts[1] != "-777" else None
                status = parts[2] if len(parts) > 2 else ''
                data.append({"date": timestamp, "value": value, "status": status})
    
    # Create DataFrame
    df = pd.DataFrame(data)
    if not df.empty:
        df.set_index("date", inplace=True)
    
    return df

In [4]:
def parse_station_data(id: str, aggregate_hourly: bool, zrx_discharge_path: str = None, zrx_water_level_path: str = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse station data for a given station ID, with an option to aggregate hourly.
    Data before 2024 is already in UTC+0 timezone, only convert data after 2024.
    
    Args:
        id (str): Station ID
        aggregate_hourly (bool): Whether to aggregate data to hourly intervals
        zrx_discharge_path (str, optional): Path to the discharge ZRX file for data after 2024
        zrx_water_level_path (str, optional): Path to the water level ZRX file for data after 2024
    
    """
    # Construct file path for old data
    f_data = glob(str(input_path / "Q_and_W" / f"{id}*" / f"{id}*.csv"))[0]

    # Read old data
    data = pd.read_csv(f_data, encoding="latin1", sep=";", decimal=",")

    # Rename columns
    data.rename(columns={"Zeitpunkt": "date", 
                         "Durchfluss (15 Minuten) [m³/s]": "discharge_vol_obs", 
                         "Wasserstand (15 Minuten) [cm]": "water_level_obs"}, inplace= True)
    
    # Time is missing at midnight (e.g. "01.01.2007" instead of "01.01.2007 00:00:00")
    data["date"] = data["date"].apply(lambda x: x if len(x) == 19 else x + " 00:00:00")
    data["date"] = pd.to_datetime(data["date"], format="%d.%m.%Y %H:%M:%S")
    
    # Set timezone information (data is already in UTC)
    data["date"] = data["date"].dt.tz_localize("UTC")
    data = data.set_index("date")

    # Read and merge discharge ZRX data if file path is provided
    if zrx_discharge_path:
        zrx_discharge_data = parse_data_after_2024(zrx_discharge_path, id)
        
        if not zrx_discharge_data.empty:
            # Convert ZRX data to match old data format
            zrx_discharge_formatted = pd.DataFrame(index=zrx_discharge_data.index)
            zrx_discharge_formatted["discharge_vol_obs"] = zrx_discharge_data["value"]
            zrx_discharge_formatted["water_level_obs"] = None
            
            # Set timezone for ZRX data (assuming UTC+1 as stated in the header)
            zrx_discharge_formatted.index = zrx_discharge_formatted.index.tz_localize("Etc/GMT-1").tz_convert("UTC")
            
            # Filter ZRX data to only include dates after the old data
            if not data.empty:
                last_old_date = data.index.max()
                zrx_discharge_new = zrx_discharge_formatted[zrx_discharge_formatted.index > last_old_date]
            else:
                zrx_discharge_new = zrx_discharge_formatted
        else:
            zrx_discharge_new = pd.DataFrame()
    else:
        zrx_discharge_new = pd.DataFrame()

    # Read and merge water level ZRX data if file path is provided
    if zrx_water_level_path:
        zrx_water_level_data = parse_data_after_2024(zrx_water_level_path, id)
        
        if not zrx_water_level_data.empty:
            # Convert ZRX data to match old data format
            zrx_water_level_formatted = pd.DataFrame(index=zrx_water_level_data.index)
            zrx_water_level_formatted["discharge_vol_obs"] = None
            zrx_water_level_formatted["water_level_obs"] = zrx_water_level_data["value"]
            
            # Set timezone for ZRX data (assuming UTC+1 as stated in the header)
            zrx_water_level_formatted.index = zrx_water_level_formatted.index.tz_localize("Etc/GMT-1").tz_convert("UTC")
            
            # Filter ZRX data to only include dates after the old data
            if not data.empty:
                last_old_date = data.index.max()
                zrx_water_level_new = zrx_water_level_formatted[zrx_water_level_formatted.index > last_old_date]
            else:
                zrx_water_level_new = zrx_water_level_formatted
        else:
            zrx_water_level_new = pd.DataFrame()
    else:
        zrx_water_level_new = pd.DataFrame()

    # Combine all new ZRX data
    if not zrx_discharge_new.empty or not zrx_water_level_new.empty:
        # Create a combined index from both datasets
        all_timestamps = set()
        if not zrx_discharge_new.empty:
            all_timestamps.update(zrx_discharge_new.index)
        if not zrx_water_level_new.empty:
            all_timestamps.update(zrx_water_level_new.index)
        
        # Create combined dataframe with all timestamps
        zrx_combined = pd.DataFrame(index=sorted(all_timestamps))
        zrx_combined["discharge_vol_obs"] = None
        zrx_combined["water_level_obs"] = None
        
        # Fill discharge values
        if not zrx_discharge_new.empty:
            zrx_combined.loc[zrx_discharge_new.index, "discharge_vol_obs"] = zrx_discharge_new["discharge_vol_obs"]
        
        # Fill water level values  
        if not zrx_water_level_new.empty:
            zrx_combined.loc[zrx_water_level_new.index, "water_level_obs"] = zrx_water_level_new["water_level_obs"]
        
        # Combine old and new data
        data = pd.concat([data, zrx_combined], sort=False)

        # Name index
        data.index.name = "date"
        
    # Reset index for aggregation if needed
    data = data.reset_index()
    
    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # Metadata
    f_meta = glob(str(input_path / "Q_and_W" / f"{id}*" / "Metadaten.csv"))[0]

    meta = pd.read_csv(f_meta, encoding="latin1", sep=";")

    # format the metadata
    meta = meta.T
    meta.columns = meta.iloc[0]
    meta = meta.iloc[1:]
    if "Einzugsgebietsgröße" in meta.columns:
        meta["Einzugsgebietsgröße"] = meta["Einzugsgebietsgröße"].str.replace(" km²", "").str.replace(",", ".").astype(float)
    else:
        meta["Einzugsgebietsgröße"] = None

    return meta, data

parse_station_data(id="440004", aggregate_hourly=True, zrx_discharge_path=str(input_path / "Daten2024_2025" / "Q.zrx"), zrx_water_level_path=str(input_path / "Daten2024_2025" / "W.zrx"))[1]

,date,discharge_vol_obs,water_level_obs
0,2007-01-01 00:00:00+00:00,0.13,12.0
1,2007-01-01 01:00:00+00:00,0.13,12.0
2,2007-01-01 02:00:00+00:00,0.13,12.0
3,2007-01-01 03:00:00+00:00,0.13,12.0
4,2007-01-01 04:00:00+00:00,0.13,12.0
...,...,...,...
163027,2025-08-06 19:00:00+00:00,0.043,15.0
163028,2025-08-06 20:00:00+00:00,0.043,15.0
163029,2025-08-06 21:00:00+00:00,0.043,15.0
163030,2025-08-06 22:00:00+00:00,0.043,15.0


The catchment area information can be different between the CAMELS-DE (camelsp) metadata and the newly provided metadata.  
Often, differences are very small but are large for some catchments.  
To be in line with the CAMELS-DE catchment area information, we will use the CAMELS-DE catchment area information.

**but this can be discussed / changed later**

In [5]:
def check_metadata_consistency(meta, camelsp_meta, raw_meta, id: str) -> dict:
    """
    Check consistency of metadata fields across the different metadata sources.

    """
    inconsistencies = {}
    
    # Define field mappings between sources
    fields = {
        "provider_id": {
            "meta": "Pegelkennziffer",
            "camelsp": "provider_id",
            "raw": "kennziffer",
            "transform_raw": lambda x: str(x)
        },
        "gauge_name": {
            "meta": "Pegelname",
            "camelsp": "gauge_name",
            "raw": "pegelname"
        },
        "waterbody": {
            "meta": "Gewässer",
            "camelsp": "waterbody_name",
            "raw": "gewaesser"
        },
        "area": {
            "meta": "Einzugsgebietsgröße",
            "camelsp": "area",
            "raw": "ezg_km2"
        }
    }

    if camelsp_meta.empty:
        # Two-way comparisons
        for field_key, field_info in fields.items():
            meta_val = meta[field_info["meta"]].values[0]
            raw_val = raw_meta[field_info["raw"]].values[0]
            
            # Apply transform if specified
            if "transform_raw" in field_info:
                raw_val = field_info["transform_raw"](raw_val)
                
            if meta_val != raw_val:
                if id in inconsistencies:
                    inconsistencies[id].update({field_key: {
                        "meta": meta_val,
                        "raw": raw_val
                    }})
                else:
                    inconsistencies[id] = {field_key: {
                        "meta": meta_val,
                        "raw": raw_val
                    }}
    else:
        # Three-way comparisons
        for field_key, field_info in fields.items():
            meta_val = meta[field_info["meta"]].values[0]
            camelsp_val = camelsp_meta[field_info["camelsp"]].values[0]
            raw_val = raw_meta[field_info["raw"]].values[0]
            
            # Apply transform if specified
            if "transform_raw" in field_info:
                raw_val = field_info["transform_raw"](raw_val)
                
            if not (meta_val == camelsp_val == raw_val):
                if id in inconsistencies:
                    inconsistencies[id].update({field_key: {
                        "meta": meta_val,
                        "camelsp": camelsp_val,
                        "raw": raw_val
                    }})
                else:
                    inconsistencies[id] = {field_key: {
                        "meta": meta_val,
                        "camelsp": camelsp_val,
                        "raw": raw_val
                    }}
    
    return inconsistencies

In [6]:
ids = sorted([f.split("/")[-1].split(",")[0] for f in glob(str(input_path / "Q_and_W" / "*"))])

# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

# for Sachsen-Anhalt there is a raw metadata file that we can use if the station is not in the CAMELS-DE metadata
raw_meta_all = gpd.read_file(input_path / "Pegel Wasserstand-Durchfluss (gepruefte Daten)" / "pegel.gpkg")
raw_meta_all["kennziffer"] = raw_meta_all["kennziffer"].astype(str)
# make column ezg_km2 float (732,10 km² -> 732.10)
raw_meta_all["ezg_km2"] = raw_meta_all["ezg_km2"].str.replace(" km²", "").str.replace(",", ".").astype(float)
raw_meta_all["hoehe"] = raw_meta_all["hoehe"].astype(float)

metadata_inconsistencies = {}

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DE2", add_missing=True)

    # parse metadata and data for the station
    meta, data = parse_station_data(id, aggregate_hourly=True)

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["kennziffer"] == id]

    # check the consistency of the metadata
    inconsistencies = check_metadata_consistency(meta, camelsp_meta, raw_meta, id)
    if inconsistencies:
        metadata_inconsistencies.update(inconsistencies)

# Create dictionaries for each source
meta_areas = {k: v['area']['meta'] for k, v in metadata_inconsistencies.items()}
camelsp_areas = {k: v['area']['camelsp'] for k, v in metadata_inconsistencies.items()}
raw_areas = {k: v['area']['raw'] for k, v in metadata_inconsistencies.items()}

# Create DataFrame
metadata_inconsistencies_df = pd.DataFrame({
    'meta': meta_areas,
    'camelsp': camelsp_areas,
    'raw': raw_areas
})

# Sort index
metadata_inconsistencies_df.sort_index(inplace=True)

# calculate difference between meta and camelsp columns
metadata_inconsistencies_df['meta-camelsp'] = metadata_inconsistencies_df['meta'] - metadata_inconsistencies_df['camelsp']
metadata_inconsistencies_df.sort_values(by='meta-camelsp', ascending=False)

100%|██████████| 97/97 [03:17<00:00,  2.04s/it]


,meta,camelsp,raw,meta-camelsp
560090,6990.00,1655.00,6990.00,5335.00
5898807,499.80,57.80,499.80,442.00
570611,12085.00,12053.00,12085.00,32.00
594010,1597.00,1572.00,1597.00,25.00
578600,681.00,658.00,681.00,23.00
576240,119.30,99.00,119.30,20.30
594104,154.00,136.00,154.00,18.00
573360,6218.00,6201.00,6218.00,17.00
579049,1215.00,1201.00,1215.00,14.00
570340,5040.00,5026.00,5040.00,14.00


## Parse data and metadata for the stations and save to ouput directory

We priotiize the CAMELS-DE (camelsp) metadata information over the newly provided metadata information to be in line with the CAMELS-DE catchment area information.

In [6]:
ids = sorted([f.split("/")[-1].split(",")[0] for f in glob(str(input_path / "Q_and_W" / "*"))])

# get the metadata from CAMELS-DE daily
camelsp_meta_all = get_metadata()

# for Sachsen-Anhalt there is a raw metadata file that we can use if the station is not in the CAMELS-DE metadata
raw_meta_all = gpd.read_file(input_path / "Pegel Wasserstand-Durchfluss (gepruefte Daten)" / "pegel.gpkg")
raw_meta_all["kennziffer"] = raw_meta_all["kennziffer"].astype(str)
# make column ezg_km2 float (732,10 km² -> 732.10)
raw_meta_all["ezg_km2"] = raw_meta_all["ezg_km2"].str.replace(" km²", "").str.replace(",", ".").astype(float)
raw_meta_all["hoehe"] = raw_meta_all["hoehe"].astype(float)

for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DEE", add_missing=True)

    # parse metadata and data for the station
    meta, data = parse_station_data(id, aggregate_hourly=True, zrx_discharge_path=str(input_path / "Daten2024_2025" / "Q.zrx"), zrx_water_level_path=str(input_path / "Daten2024_2025" / "W.zrx"))

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["kennziffer"] == id]

    if not camelsp_meta.empty:
        # set metadata values from camelsp metadata
        station_meta = {
            "gauge_id": camelsp_meta["camels_id"].values[0],
            "provider_id": camelsp_meta["provider_id"].values[0],
            "gauge_name": camelsp_meta["gauge_name"].values[0],
            "waterbody_name": camelsp_meta["waterbody_name"].values[0],
            "federal_state": camelsp_meta["federal_state"].values[0],
            "gauge_lon": camelsp_meta["lon"].values[0],
            "gauge_lat": camelsp_meta["lat"].values[0],
            "gauge_easting": camelsp_meta["x"].values[0],
            "gauge_northing": camelsp_meta["y"].values[0],
            "gauge_elev_metadata": camelsp_meta["gauge_elevation"].values[0],
            "area_metadata": camelsp_meta["area"].values[0],
            "part_of_camelsp": True,
        }
    elif not raw_meta.empty:
        # if the station is not in the camelsp metadata, use the raw metadata
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": raw_meta["pegelname"].values[0],
            "waterbody_name": raw_meta["gewaesser"].values[0],
            "federal_state": "Sachsen-Anhalt",
            "gauge_lon": raw_meta.to_crs("epsg:4326").geometry.x.values[0],
            "gauge_lat": raw_meta.to_crs("epsg:4326").geometry.y.values[0],
            "gauge_easting": raw_meta.to_crs("epsg:3035").geometry.x.values[0],
            "gauge_northing": raw_meta.to_crs("epsg:3035").geometry.y.values[0],
            "gauge_elev_metadata": raw_meta["hoehe"].values[0],
            "area_metadata": raw_meta["ezg_km2"].values[0],
            "part_of_camelsp": False,
        }
    else:
        raise ValueError(f"Station {id} not found in CAMELS-DE metadata or raw metadata.")
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(pd.concat([meta.reset_index(drop=True), raw_meta.reset_index(drop=True)], axis=1))
    

 29%|██▉       | 28/97 [07:08<16:49, 14.62s/it]/tmp/ipykernel_568803/3289561425.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data, zrx_combined], sort=False)
100%|██████████| 97/97 [22:50<00:00, 14.13s/it]
